# Data audit

Inspect file structure, subject coverage, and quality flags. Set `COHORTS` to select years; edit `DATA_DIR` below to point to the data folder.

Requires Python 3.10+, NumPy, pandas, IPython.

In [1]:
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Iterable, List
import re
import numpy as np
import pandas as pd
from IPython.display import display

# Data directory containing the 1year, 2year, and 3year folders.
DATA_DIR = Path(r"C:\Users\emanu\Documents\Progetti_python\SLD-slimGSGP\data")
COHORTS = None  # All years; use ["1year"] to select one cohort.

EDA_VALID_LABELS = (0, 1, 2)
EDA_UNKNOWN_LABEL = -1
EDA_EXPECTED_TASKS = {"1year": 21, "2year": 21, "3year": 18}
AGE_MIN_VALID = 3.0
AGE_MAX_VALID = 20.0
GENDER_VALUES = ["Femminile", "Maschile"]
DOMINANT_HAND_VALUES = ["Destra", "Sinistra"]

In [2]:
@dataclass
class TaskTable:
    """One source file; source is relative to its cohort directory."""

    cohort: str
    source: str
    frame: pd.DataFrame

    @property
    def task(self) -> str:
        return str(Path(self.source).with_suffix(""))


def natural_key(value: str) -> tuple:
    """Sort numeric filename components numerically, with deterministic ties."""
    return tuple(
        (1, int(part)) if part.isdigit() else (0, part.casefold())
        for part in re.split(r"(\d+)", value)
    ) + ((0, value),)


def load_tasks(data_dir: Path, cohorts: Iterable[str] | None = None) -> list[TaskTable]:
    """Discover CSVs in natural order; preserve missing IDs and unknown labels."""
    root = Path(data_dir).expanduser()
    if not root.is_dir():
        raise FileNotFoundError(
            f"Set DATA_DIR to the data directory containing cohort folders: {root}"
        )
    selected = (
        list(cohorts)
        if cohorts is not None
        else [
            p.name for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")
        ]
    )
    if len(selected) != len(set(selected)):
        raise ValueError("Select each cohort only once.")
    tables = []
    for cohort in sorted(selected, key=natural_key):
        directory = root / cohort
        if not directory.is_dir():
            raise FileNotFoundError(f"Cohort directory does not exist: {cohort}")
        files = sorted(
            (
                p
                for p in directory.rglob("*")
                if p.is_file() and p.suffix.lower() == ".csv"
            ),
            key=lambda p: natural_key(p.relative_to(directory).as_posix()),
        )
        if not files:
            raise FileNotFoundError(f"No task CSVs found in cohort: {cohort}")
        for path in files:
            frame = pd.read_csv(path, dtype="string")
            tables.append(
                TaskTable(cohort, path.relative_to(directory).as_posix(), frame)
            )
    if not tables:
        raise ValueError("No cohorts selected.")
    return tables


def cohort_tables(tables: list[TaskTable]) -> dict[str, list[TaskTable]]:
    result: dict[str, list[TaskTable]] = {}
    for table in tables:
        result.setdefault(table.cohort, []).append(table)
    return result


tables = load_tasks(DATA_DIR, cohorts=COHORTS)
pd.set_option("display.max_rows", 24)
pd.set_option("display.max_columns", 12)
print(
    "Loaded task files by cohort:",
    {c: len(g) for c, g in cohort_tables(tables).items()},
)

Loaded task files by cohort: {'1year': 21, '2year': 21, '3year': 18}


IDs are cohort-local. Task-count expectations are editable in the configuration cell.

In [3]:
@dataclass
class AuditResult:
    inventory: pd.DataFrame
    schema: pd.DataFrame
    flags: pd.DataFrame
    coverage: pd.DataFrame
    cohorts: pd.DataFrame


EVENT_FEATURES: List[str] = [
    "number_of_interruptions",
]


KINEMATICS_FEATURES: List[str] = [
    "mean:velocity:axis-xy(on-surface)",
    "std:velocity:axis-xy(on-surface)",
    "mean:acceleration:axis-xy(on-surface)",
    "std:acceleration:axis-xy(on-surface)",
    "mean:jerk:axis-xy(on-surface)",
    "average_normalized_jerk",
]


PRESSURE_FEATURES: List[str] = [
    "mean:pressure",
    "std:pressure",
    "iqr:pressure",
    "number_of_changes_in_pressure_profile",
]


SPATIAL_FEATURES: List[str] = [
    "writing_width(on-surface)",
    "writing_height(on-surface)",
    "mean:stroke_width(on-surface)",
    "std:stroke_width(on-surface)",
    "iqr:stroke_width(on-surface)",
    "mean:stroke_height(on-surface)",
    "std:stroke_height(on-surface)",
    "iqr:stroke_height(on-surface)",
    "mean:stroke_length(on-surface)",
    "number_of_changes_in_x_profile",
    "number_of_changes_in_y_profile",
]


TEMPORAL_FEATURES: List[str] = [
    "writing_duration(on-surface)",
    "writing_duration_overall",
    "mean:stroke_duration(on-surface)",
    "in_air_movement_time",
    "in_air_on_paper_ratio",
]


TILT_FEATURES: List[str] = [
    "mean:tilt(on-surface)",
    "std:tilt(on-surface)",
    "iqr:tilt(on-surface)",
]


KINEMATIC_FEATURES: List[str] = (
    PRESSURE_FEATURES
    + TILT_FEATURES
    + TEMPORAL_FEATURES
    + SPATIAL_FEATURES
    + KINEMATICS_FEATURES
    + EVENT_FEATURES
)


TARGET_COLUMN: str = "Label"


CSV_DTYPES: Dict[str, str] = {
    "Id": "int64",
    **{f: "float64" for f in KINEMATIC_FEATURES},
    "Gender": "str",
    "Age": "float64",
    "Dominant_Hand": "str",
    TARGET_COLUMN: "int64",
    "Task": "int64",
}


DEMOGRAPHIC_COLUMNS: List[str] = [
    "Gender",
    "Age",
    "Dominant_Hand",
]


def filename_task_id(source: str) -> int | None:
    match = re.fullmatch(r"Task_(\d+)", Path(source).stem, flags=re.IGNORECASE)
    return int(match.group(1)) if match else None


def measurement_masks(values: pd.DataFrame) -> dict[str, pd.Series]:
    """Zero-only requires every measurement to be present and exactly zero."""
    finite = pd.DataFrame(
        np.isfinite(values), index=values.index, columns=values.columns
    )
    if values.shape[1] == 0:
        return {
            name: pd.Series(name == "no_finite_measurements", index=values.index)
            for name in ("all_zero_measurements", "no_finite_measurements")
        }
    return {
        "all_zero_measurements": values.eq(0).all(axis=1) & finite.all(axis=1),
        "no_finite_measurements": ~finite.any(axis=1),
    }


def numerical_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Include known measurements plus numerical extra columns; exclude metadata."""
    metadata = set(CSV_DTYPES) - {
        c for c, dtype in CSV_DTYPES.items() if dtype == "float64" and c != "Age"
    }
    columns = []
    for column in frame.columns:
        if column in metadata or column == "Cohort":
            continue
        converted = pd.to_numeric(frame[column], errors="coerce")
        if column in CSV_DTYPES or converted.notna().any():
            columns.append(column)
    return frame[columns].apply(pd.to_numeric, errors="coerce").astype(float)


def audit_tasks(tables: list[TaskTable]) -> AuditResult:
    """Keep flags aggregate: no subject identifiers or raw records are exported."""
    inventory, schema, flags, coverage, summaries = [], [], [], [], []

    def flag(cohort: str, task: str, issue: str, count: int, detail: str = "") -> None:
        if count:
            flags.append(
                dict(
                    cohort=cohort,
                    task=task,
                    issue=issue,
                    count=int(count),
                    detail=detail,
                )
            )

    for cohort, group in cohort_tables(tables).items():
        sets = []
        for table in group:
            ids = table.frame.get("Id", pd.Series(dtype="string")).dropna()
            sets.append(set(ids[ids.astype("string").str.strip().ne("")]))
        union = set().union(*sets)
        common = set.intersection(*sets)
        expected = EDA_EXPECTED_TASKS.get(cohort)
        if expected is not None and len(group) != expected:
            flag(
                cohort,
                "all",
                "unexpected_task_count",
                1,
                f"Observed {len(group)} files; configured expectation is {expected}.",
            )
        declarations: dict[float, list[str]] = {}
        for table, subjects in zip(group, sets):
            df, task = table.frame, table.task
            values = numerical_features(df)
            masks = measurement_masks(values)
            ids = df.get("Id", pd.Series(pd.NA, index=df.index, dtype="string"))
            blank_ids = ids.isna() | ids.astype("string").str.strip().eq("").fillna(
                False
            )
            duplicates = ids[~blank_ids].duplicated(keep=False).sum()
            inventory.append(
                dict(
                    cohort=cohort,
                    task=task,
                    rows=len(df),
                    unique_subjects=len(subjects),
                    columns=len(df.columns),
                    features=values.shape[1],
                    missing_ids=int(blank_ids.sum()),
                    duplicate_id_rows=int(duplicates),
                    nan_cells=int(df.isna().sum().sum()),
                    infinite_feature_cells=int(np.isinf(values.to_numpy()).sum()),
                    all_zero_rows=int(masks["all_zero_measurements"].sum()),
                    no_finite_measurement_rows=int(
                        masks["no_finite_measurements"].sum()
                    ),
                )
            )
            coverage.append(
                dict(
                    cohort=cohort,
                    task=task,
                    subjects=len(subjects),
                    union_subjects=len(union),
                    common_subjects=len(common),
                    missing_from_union=len(union - subjects),
                    matches_first_task=subjects == sets[0],
                )
            )
            flag(cohort, task, "missing_id", blank_ids.sum())
            flag(cohort, task, "duplicate_id_rows", duplicates)
            flag(cohort, task, "incomplete_subject_coverage", len(union - subjects))
            flag(cohort, task, "nan_cells", df.isna().sum().sum())
            flag(
                cohort,
                task,
                "infinite_feature_cells",
                np.isinf(values.to_numpy()).sum(),
            )
            for issue, mask in masks.items():
                flag(cohort, task, issue, mask.sum())
            for column in sorted(set(CSV_DTYPES) | set(df.columns)):
                present = column in df
                expected_type = CSV_DTYPES.get(column, "unrecognized_column")
                invalid = 0
                if present and expected_type in ("int64", "float64"):
                    parsed = pd.to_numeric(df[column], errors="coerce")
                    invalid = (df[column].notna() & parsed.isna()).sum()
                    if expected_type == "int64":
                        invalid += (
                            parsed.notna()
                            & (~np.isfinite(parsed) | parsed.mod(1).ne(0))
                        ).sum()
                schema.append(
                    dict(
                        cohort=cohort,
                        task=task,
                        column=column,
                        present=present,
                        expected_dtype=expected_type,
                        invalid_numeric_cells=int(invalid),
                    )
                )
                flag(cohort, task, "missing_column", int(not present), column)
                flag(
                    cohort,
                    task,
                    "unrecognized_column",
                    int(column not in CSV_DTYPES),
                    column,
                )
                flag(cohort, task, "invalid_numeric_cells", invalid, column)
            task_values = pd.to_numeric(
                df.get("Task", pd.Series(dtype=float)), errors="coerce"
            )
            declared = task_values.dropna().unique()
            for value in declared:
                declarations.setdefault(value, []).append(task)
            filename_id = filename_task_id(table.source)
            flag(cohort, task, "unrecognized_task_filename", int(filename_id is None))
            flag(cohort, task, "multiple_task_values", int(len(declared) > 1))
            if filename_id is not None and "Task" in df:
                flag(
                    cohort,
                    task,
                    "task_filename_mismatch",
                    (task_values.ne(filename_id) | task_values.isna()).sum(),
                )
            if {"Id", "Task"}.issubset(df):
                flag(
                    cohort,
                    task,
                    "duplicate_subject_task_rows",
                    df.loc[~blank_ids, ["Id", "Task"]].duplicated(keep=False).sum(),
                )
            if "Label" in df:
                labels = pd.to_numeric(df.Label, errors="coerce")
                flag(
                    cohort,
                    task,
                    "unknown_label_minus_one",
                    labels.eq(EDA_UNKNOWN_LABEL).sum(),
                    "Meaning unresolved; retained separately from classification labels.",
                )
                flag(
                    cohort,
                    task,
                    "unexpected_or_missing_label",
                    (~labels.isin((*EDA_VALID_LABELS, EDA_UNKNOWN_LABEL))).sum(),
                )
            if "Age" in df:
                age = pd.to_numeric(df.Age, errors="coerce")
                invalid = ~age.between(
                    AGE_MIN_VALID, AGE_MAX_VALID, inclusive="neither"
                )
                for value, count in (
                    df.loc[invalid, "Age"].value_counts(dropna=False).items()
                ):
                    flag(
                        cohort,
                        task,
                        "implausible_or_missing_age",
                        count,
                        f"Age={value!r}",
                    )
            for column, allowed in (
                ("Gender", GENDER_VALUES),
                ("Dominant_Hand", DOMINANT_HAND_VALUES),
            ):
                if column in df:
                    invalid = ~df[column].isin(allowed)
                    for value, count in (
                        df.loc[invalid, column].value_counts(dropna=False).items()
                    ):
                        flag(
                            cohort,
                            task,
                            "unrecognized_or_missing_category",
                            count,
                            f"{column}={value!r}",
                        )
        for value, sources in declarations.items():
            flag(
                cohort,
                "all",
                "task_identifier_in_multiple_files",
                int(len(sources) > 1),
                f"Declared task {value:g} occurs in {len(sources)} files.",
            )
        combined = pd.concat(
            [t.frame.assign(_source=t.source) for t in group], ignore_index=True
        )
        if {"Id", "Task"}.issubset(combined):
            source_counts = combined.groupby(["Id", "Task"])["_source"].transform(
                "nunique"
            )
            flag(
                cohort,
                "all",
                "cross_file_duplicate_subject_task_rows",
                source_counts.gt(1).sum(),
            )
        conflicts = {}
        for column in DEMOGRAPHIC_COLUMNS:
            count = 0
            if {"Id", column}.issubset(combined):
                count = int(
                    combined.groupby("Id")[column].nunique(dropna=False).gt(1).sum()
                )
            conflicts[column] = count
            flag(cohort, "all", "cross_task_demographic_conflict", count, column)
        summaries.append(
            dict(
                cohort=cohort,
                task_files=len(group),
                expected_tasks=expected,
                rows=sum(len(t.frame) for t in group),
                subjects=len(union),
                subjects_in_every_task=len(common),
                identical_subject_sets=all(s == sets[0] for s in sets),
                **{f"{c}_conflicting_subjects": n for c, n in conflicts.items()},
            )
        )
    return AuditResult(
        pd.DataFrame(inventory),
        pd.DataFrame(schema),
        pd.DataFrame(flags, columns=["cohort", "task", "issue", "count", "detail"]),
        pd.DataFrame(coverage),
        pd.DataFrame(summaries),
    )


audit = audit_tasks(tables)
display(audit.cohorts)

,cohort,task_files,expected_tasks,rows,subjects,subjects_in_every_task,identical_subject_sets,Gender_conflicting_subjects,Age_conflicting_subjects,Dominant_Hand_conflicting_subjects
0,1year,21,21,2935,140,136,False,0,0,0
1,2year,21,21,3045,145,145,True,0,0,0
2,3year,18,18,2610,145,145,True,0,0,0


In [4]:
inventory_columns = [
    "cohort",
    "task",
    "rows",
    "unique_subjects",
    "columns",
    "features",
    "all_zero_rows",
]
display(audit.inventory[inventory_columns].set_index(["cohort", "task"]))
print("Every discovered CSV is represented:", len(audit.inventory) == len(tables))

rows  unique_subjects  columns  features  all_zero_rows
cohort task                                                            
1year  Task_1    140              140       36        30              1
       Task_2    140              140       36        30              1
       Task_3    140              140       36        30              1
       Task_4    140              140       36        30              1
       Task_5    140              140       36        30              1
...              ...              ...      ...       ...            ...
3year  Task_14   145              145       36        30             25
       Task_15   145              145       36        30             25
       Task_16   145              145       36        30             25
       Task_17   145              145       36        30             25
       Task_18   145              145       36        30             25

[60 rows x 5 columns]

Every discovered CSV is represented: True


Matching demographics support subject linkage; confirm the acquisition IDs. Missing records, zero-only rows, and nonfinite measurements require separate checks.

In [5]:
display(audit.coverage.loc[audit.coverage.missing_from_union.gt(0)])
display(
    audit.flags.groupby(["cohort", "issue"], sort=False)["count"]
    .sum()
    .unstack(fill_value=0)
    .T
)

,cohort,task,subjects,union_subjects,common_subjects,missing_from_union,matches_first_task
19,1year,Task_20,139,140,136,1,False
20,1year,Task_21,136,140,136,4,False


cohort,1year,2year,3year
issue,,,
all_zero_measurements,21,547,450
unknown_label_minus_one,141,666,1410
unrecognized_or_missing_category,42,1092,900
incomplete_subject_coverage,5,0,0


In [6]:
schema_issues = audit.schema.loc[
    ~audit.schema.present
    | audit.schema.invalid_numeric_cells.gt(0)
    | audit.schema.expected_dtype.eq("unrecognized_column")
]
display(schema_issues.head(20))
print("Schema issue entries:", len(schema_issues))
display(
    audit.flags.loc[
        audit.flags.issue.isin(
            [
                "task_filename_mismatch",
                "multiple_task_values",
                "task_identifier_in_multiple_files",
                "missing_id",
                "duplicate_id_rows",
                "cross_task_demographic_conflict",
            ]
        )
    ].head(20)
)

display(
    audit.flags.loc[
        audit.flags.issue.isin(
            ["implausible_or_missing_age", "unrecognized_or_missing_category"]
        )
    ]
    .groupby(["cohort", "issue", "detail"], sort=False)["count"]
    .sum()
    .to_frame("task_rows")
)

,cohort,task,column,present,expected_dtype,invalid_numeric_cells


Schema issue entries: 0


,cohort,task,issue,count,detail


task_rows
cohort issue                            detail                      
1year  unrecognized_or_missing_category Gender='0'                21
                                        Dominant_Hand='0'         21
2year  unrecognized_or_missing_category Gender='0'               546
                                        Dominant_Hand='0'        546
3year  unrecognized_or_missing_category Gender='0'               450
                                        Dominant_Hand='0'        450